In [ ]:
# Just some initial setup
import pandas as pd
import plotly.express as px
from pathlib import Path

cities = ['Chicago', 'Philadelphia', 'Boston']
colors = {'Chicago': '#0072B2', 'Philadelphia': '#D55E00', 'Boston': '#009E73'}

In [ ]:
# Change this if the repo is somewhere else on your computer, this is just where its at for me
repo_path = Path.home() / "Documents" / "DATA303_Final"
data_path = repo_path / "Data"

commutes = pd.read_csv(data_path / "commutes_2025.csv").set_index("label")[cities]
vehicles = pd.read_csv(data_path / "vehicle_ownership_2025.csv").set_index("label")[cities]
metrics = pd.read_csv(data_path / "stop_density_etc_metrics.csv").set_index("city").loc[cities]

# The mean is in minutes, so it doesn't belong with the percentage bins
commute_bins = commutes.drop(index="Mean travel time to work (minutes)").copy()
commute_bins.index = commute_bins.index.str.replace(" (%)", "", regex=False)

# Compare shares of households instead of raw city sizes
vehicle_counts = vehicles.drop(index="Total Households")
vehicle_pct = vehicle_counts.div(vehicles.loc["Total Households"], axis=1) * 100

metrics.round(1)

,area_sq_mi,active_stops,stops_per_sq_mi,pct_no_vehicle,mean_commute_mins
city,,,,,
Chicago,227.7,10095,44.3,26.7,33.6
Philadelphia,134.3,8076,60.1,32.0,33.5
Boston,48.3,1937,40.1,36.0,32.2


## How long are peoples' commutes?

This graph uses separate bar charts to compare the distribution of commute times in Chicago, Philadelphia, and Boston. I wanted to visualize this because an average alone doesn’t show how many people have really short or long commutes. Showing the cities side by side makes those differences easier to see.

In [ ]:
# Put the city columns into one column 
commute_long = commute_bins.reset_index().melt(
    id_vars='label', var_name='City', value_name='Percent')

# Bar heights show percentages, with one panel for each city
fig_commute = px.bar(
    commute_long, x='label', y='Percent', color='City', facet_col='City', # Split the chart into city panels
    color_discrete_map=colors, # Apply the colors we assigned
    category_orders={'City': cities, 'label': list(commute_bins.index)},
    labels={'label': 'Commute time', 'Percent': 'Share of commuters (%)'},
    # Show percentage rounded to one decimal and also don't show city on hover since it is redundant
    hover_data={'City': False, 'Percent': ':.1f'}, 
    template='plotly_white', height=550
)

# Set the title for our visualization
fig_commute.update_layout(
    title='30–34 minutes is the most common commute time interval in all three cities',
    showlegend=False, font={'family': 'Arial', 'size': 13},
    margin={'t': 150, 'b': 150}
)

# Use the same scale for each city, fixedrange makes it so zoom is diabled, we don't need it, it also can mess with the visualization
fig_commute.update_yaxes(range=[0, 22], ticksuffix='%', fixedrange=True)
fig_commute.update_xaxes(tickangle=-55, fixedrange=True, title_text='')
# One shared label underneath the three panels.
fig_commute.add_annotation(
    text='Commute time', x=0.5, y=-0.50, # Centered, below plot
    xref='paper', yref='paper', showarrow=False,
    xanchor='center', yanchor='top'
)

# Extra feature, gray out bins below the selected cutoff, dropdown menu in plotly
# Each number is the row where that cutoff starts, counting from 0
buttons = []
cutoffs = {'All commute times': 0, '30+ minutes': 5, '45+ minutes': 7, '60+ minutes': 8}

# Create the dropdown option for each of the cutoffs
for label, start in cutoffs.items():
    bar_colors = []
    for city in cities:
        shades = []
        # Keep selected bins, shade out the rest in gray to minimize distraction
        for i in range(len(commute_bins)):
            if i >= start:
                shades.append(colors[city])
            else:
                shades.append('#D9D9D9')
        bar_colors.append(shades) # Save city's list of bar colors

    title = '30–34 minutes is the most common commute time interval in all three cities'
    if start > 0:
        # add percentages from the cutoff onward
        totals = commute_bins.iloc[start:].sum()
        lowest = totals.idxmin() # city with the smallest total
        title = f'{lowest} has the lowest share of {label} commutes' # Jean Luc Doumont-esque title for each viz
        # Add a smaller second line with each city's percentage, rounded to 1 decimal
        title += f'<br><sup>Chicago: {totals["Chicago"]:.1f}% | Philadelphia: {totals["Philadelphia"]:.1f}% | Boston: {totals["Boston"]:.1f}%</sup>'
    # Interacting with the dropdown changes title and updates the bar colors
    buttons.append(dict(label=label, method='update', args=[
        {'marker.color': bar_colors}, {'title.text': title}]))

# Drop down positioned in top left
fig_commute.update_layout(updatemenus=[dict(
    buttons=buttons, x=0, y=1.25, xanchor='left', yanchor='top')])

# Tweak title to be positioned nicely at the top
fig_commute.update_layout(
    title_y=0.95,
    title_yanchor='top'
)

# Show plot with no unecessary toolbar or zoom
fig_commute.show(config={'displayModeBar': False, 'scrollZoom': False})

## How many households live without a car?

This horizontal stacked bar chart compares the percentage of households with different numbers of vehicles in each city. I wanted to see how common it is for households to have no vehicle, since that connects to our question about public transit access. Using percentages makes it easier to compare cities with different numbers of households.

In [ ]:
# Turned the city columns into rows, keeping the vehicle category as "label"
vehicle_long = vehicle_pct.reset_index().melt(
    id_vars='label', var_name='City', value_name='Percent')
counts_long = vehicle_counts.reset_index().melt(
    id_vars='label', var_name='City', value_name='Households')

# Match percentage with its respective household count so hovering shows both
vehicle_long = vehicle_long.merge(counts_long, on=['label', 'City'])

# Horizontal bars, with a different shade for each vehicle category
fig_vehicles = px.bar(
    vehicle_long, x='Percent', y='City', color='label', orientation='h',
    color_discrete_sequence=['#eee7f2', '#cbb5dc', '#9a75b5', '#704393', '#45205f'],
    category_orders={'City': cities, 'label': list(vehicle_pct.index)},
    labels={'Percent': 'Share of households (%)', 'label': 'Vehicles available'},
    hover_data={'Percent': ':.1f', 'Households': ':,.0f'},
    template='plotly_white', height=480
)

# Thin white borders help separate the stacked sections
fig_vehicles.update_traces(marker_line_color='white', marker_line_width=1)

fig_vehicles.update_layout(
    title='Boston has the highest share of households without a vehicle',
    # put the vehicle categories together in each citys bar
    barmode='stack', font={'family': 'Arial', 'size': 13},
    margin={'t': 135, 'b': 100}, legend={'orientation': 'h', 'y': -0.2} # Horizontal legend for horizontal bar chart
)

# Keep the percentage scale at 0–100, even when only one category is shown
fig_vehicles.update_xaxes(range=[0, 100], ticksuffix='%')

# Dropdown shows all 5 vehicle categories
buttons = [dict(label='All categories', method='update', args=[
    {'visible': [True] * 5},
    {'title.text': 'Boston has the highest share of households without a vehicle', 'showlegend': True}])]

# Make another dropdown option for each individual category
for i, category in enumerate(vehicle_pct.index):
    visible = [False] * 5
    visible[i] = True
    highest = vehicle_pct.loc[category].idxmax()
    title = f'{highest} has the highest share in the "{category.lower()}" category'
    # Update which category is visible and change the title to match
    buttons.append(dict(label=category, method='update', args=[
        {'visible': visible}, {'title.text': title, 'showlegend': False}]))

# Position the dropdown to the upper left of the plot
fig_vehicles.update_layout(updatemenus=[dict(
    buttons=buttons, x=0, y=1.25, xanchor='left', yanchor='top')])

# Format the title to be on top of the viz
fig_vehicles.update_layout(
    title_y=0.94,
    title_yanchor='top'
)

# Disable axes scaling so there's no accidental rescaling when interacting
fig_vehicles.update_xaxes(fixedrange=True)
fig_vehicles.update_yaxes(fixedrange=True)

# Show plot with no unecessary toolbar or zoom
fig_vehicles.show(config={'displayModeBar': False, 'scrollZoom': False})